# Assignment 3.4: Mini NeRF Rendering

In this notebook you will implement the **rendering** part of NeRF. We provide an analytic radiance field instead of asking you to train an MLP, so this problem stays lightweight and focuses on the core equations.

Implementation conventions:
- Use pixel centers when generating camera rays.
- Normalize camera-space ray directions before rotating them with `c2w[:3, :3]`.
- Sample `num_samples` points uniformly in depth between `near` and `far`.
- Compute depth as the weighted sum of sampled `t_vals` without extra normalization.

Useful formulas:
- `alpha_i = 1 - exp(-sigma_i * delta_i)`
- `T_i = prod_{j < i} (1 - alpha_j)`
- `weight_i = T_i * alpha_i`
- `C(r) = sum_i weight_i * c_i`

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from scene import get_camera_config, query_radiance_field

In [ ]:
cfg = get_camera_config()
print(cfg)

In [ ]:
def generate_rays(H, W, focal, c2w):
    # TODO:
    # 1. generate per-pixel camera-space directions
    # 2. normalize them
    # 3. rotate them into the world frame with c2w[:3, :3]
    # 4. broadcast the camera center as ray origins
    raise NotImplementedError

In [ ]:
def sample_points(rays_o, rays_d, near, far, num_samples):
    # TODO:
    # uniformly sample num_samples depth values between near and far
    # and return sampled 3D points with shape [H, W, num_samples, 3].
    raise NotImplementedError

In [ ]:
def volume_render(sigmas, rgbs, deltas, t_vals):
    # TODO:
    # alpha_i = 1 - exp(-sigma_i * delta_i)
    # T_i = product_{j < i} (1 - alpha_j)
    # weight_i = T_i * alpha_i
    # Use the weights to compute rgb and depth.
    raise NotImplementedError

In [ ]:
H, W = cfg['H'], cfg['W']
focal = cfg['focal']
near = cfg['near']
far = cfg['far']
num_samples = cfg['num_samples']
c2w = cfg['c2w']

rays_o, rays_d = generate_rays(H, W, focal, c2w)
points, t_vals, deltas = sample_points(rays_o, rays_d, near, far, num_samples)
sigmas, rgbs = query_radiance_field(points)
pred_rgb, pred_depth, weights = volume_render(sigmas, rgbs, deltas, t_vals)

print('pred_rgb shape:', pred_rgb.shape)
print('pred_depth shape:', pred_depth.shape)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.clip(pred_rgb, 0.0, 1.0))
axes[0].set_title('your render')
axes[1].imshow(pred_depth, cmap='magma')
axes[1].set_title('pred depth')
for ax in axes:
    ax.axis('off')
plt.tight_layout()

In [ ]:
opacity = np.sum(weights, axis=-1)
metrics = {
    'rgb_min': float(pred_rgb.min()),
    'rgb_max': float(pred_rgb.max()),
    'depth_min': float(pred_depth.min()),
    'depth_max': float(pred_depth.max()),
    'mean_opacity': float(opacity.mean()),
}
print(metrics)

os.makedirs('../results', exist_ok=True)
np.save('../results/mini_nerf_rgb.npy', pred_rgb)
np.save('../results/mini_nerf_depth.npy', pred_depth)
np.save('../results/mini_nerf_metrics.npy', metrics)

Expected deliverables: save `mini_nerf_rgb.npy`, `mini_nerf_depth.npy`, and `mini_nerf_metrics.npy` to `results/`. The metrics file should contain your self-check statistics, while grading should compare your outputs against a hidden reference. No training is required for this problem.